In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:

#
# import dias.rewriter
# import important libraries - matplotlib, seaborn and pandas
import pandas as pd
from pathlib import Path
from utils.benchmarks import BENCHMARKS_TO_PATHS

In [ ]:
### cell 0 ###

benchmark_name = "nyc-taxi"
file_loc = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "yellow_tripdata.csv"
factor = 1
# read file
trip_data = pd.read_csv(file_loc)
trip_data.sample(frac=factor, random_state=0)
trip_data.shape, trip_data.head()

In [ ]:
### cell 1 ###

# print data tail
trip_data.tail()

In [ ]:
### cell 2 ###

# print data info
trip_data.info()

In [ ]:
### cell 3 ###

# remove following columns - 'VendorID','RatecodeID','store_and_fwd_flag'
trip_data.drop(["VendorID", "RatecodeID", "store_and_fwd_flag"], axis=1, inplace=True)
# print data head
trip_data.head()

In [ ]:
### cell 4 ###

# convert 'tpep_pickup_datetime' and 'tpep_dropoff_datetime' to datetime on the GPU
trip_data["tpep_pickup_datetime"] = trip_data["tpep_pickup_datetime"].astype("datetime64[ns]")
trip_data["tpep_dropoff_datetime"] = trip_data["tpep_dropoff_datetime"].astype("datetime64[ns]")

# print data info and head (these are already GPU‐accelerated)
trip_data.info()
trip_data.head()

In [ ]:
### cell 5 ###

# create 'duration' column using pd.Timedelta(minutes=1)
trip_data["duration"] = (
    trip_data["tpep_dropoff_datetime"] - trip_data["tpep_pickup_datetime"]
) / pd.Timedelta(minutes=1)
# create 'trip_pickup_hour' column using 'tpep_pickup_datetime' column
trip_data["trip_pickup_hour"] = trip_data["tpep_pickup_datetime"].dt.hour
# create 'trip_dropoff_hour' column using 'tpep_dropoff_datetime' column
trip_data["trip_dropoff_hour"] = trip_data["tpep_dropoff_datetime"].dt.hour
# create 'trip_day' column using 'tpep_pickup_datetime' column - use day_name()
trip_data["trip_day"] = trip_data["tpep_pickup_datetime"].dt.day_name()
# print data info
print(trip_data.info())
# print data head
trip_data.head()

In [ ]:
### cell 6 ###

# print missing values for each column - use .isnull().sum
trip_data.isnull().sum(axis=0).reset_index()

In [ ]:
### cell 7 ###

# value_counts for 'payment_type' column
trip_data["payment_type"].value_counts()

In [ ]:
### cell 8 ###

# Vectorized mapping on GPU using cudf.Series.map and fillna
mapping = {
    1: "Credit_card",
    2: "Cash",
    3: "No_charge",
    4: "Dispute",
    5: "Unknown"
}

# Apply the mapping and fill unmapped values with "Voided_trip"
trip_data["payment_type"] = (
    trip_data["payment_type"]
    .map(mapping)
    .fillna("Voided_trip")
)

# Preview the result
trip_data.head()

In [ ]:
### cell 9 ###

# print data info to show that payment_type data type has changed
trip_data.info()

In [ ]:
### cell 10 ###

# create 'total_taxes' column from summing 'extra','mta_tax', 'improvement_surcharge'
trip_data["total_taxes"] = (
    trip_data["extra"] + trip_data["mta_tax"] + trip_data["improvement_surcharge"]
)
# drop 'extra','mta_tax','improvement_surcharge' columns
trip_data.drop(["extra", "mta_tax", "improvement_surcharge"], axis=1, inplace=True)
# print data head
trip_data.head()

In [ ]:
### cell 11 ###

# continuous_columns list
continuous_columns = [
    "fare_amount",
    "tip_amount",
    "total_taxes",
    "total_amount",
    "duration",
    "trip_distance",
    "tolls_amount",
]

In [ ]:
### cell 12 ###

# use .describe() for showing the statistics for continuous columns
trip_data[continuous_columns].describe()

In [ ]:
### cell 13 ###

# using .loc to show negative values in fare_amount
trip_data.loc[trip_data["fare_amount"] < 0]

In [ ]:
### cell 14 ###

# using .loc to show negative values in tip_amount
trip_data.loc[trip_data["tip_amount"] < 0]

In [ ]:
### cell 15 ###

# using .loc to show negative values in tolls_amount
trip_data.loc[trip_data["tolls_amount"] < 0]

In [ ]:
### cell 16 ###

# using .loc to show negative values in total_taxes
trip_data.loc[trip_data["total_taxes"] < 0]

In [ ]:
### cell 17 ###

# using .loc to show negative values in total_amount
trip_data.loc[trip_data["total_amount"] < 0]

In [ ]:
### cell 18 ###

# data shape before filtering negative fare_amount rows
print(trip_data.shape)
# using .loc to filter only those rows where fare_amount is positive
trip_data = trip_data.loc[trip_data["fare_amount"] >= 0]
trip_data.shape, trip_data.head()

In [ ]:
### cell 19 ###

# using .loc to show negative values in duration
trip_data.loc[trip_data["duration"] < 0]

In [ ]:
### cell 20 ###

# using .loc to filter only those rows where duration is positive
trip_data = trip_data.loc[trip_data["duration"] >= 0]
trip_data.shape

In [ ]:
### cell 21 ###

# use .describe() again to show the statistics for these continuous variables
trip_data[continuous_columns].describe()

In [ ]:
### cell 22 ###

# list of categorical_variables
categorical_variables = [
    "payment_type",
    "trip_pickup_hour",
    "trip_dropoff_hour",
    "trip_day",
    "PULocationID",
    "DOLocationID",
]

In [ ]:
### cell 23 ###

# start exploration with payment_type using .value_counts()
trip_data["payment_type"].value_counts()

In [ ]:
### cell 24 ###

# Use GPU-accelerated value_counts() to compute counts and convert to DataFrame
payment_type_category_count = (
    trip_data['payment_type']
    .value_counts()
    .reset_index()
)
payment_type_category_count.columns = ['payment_type', 'count']

# Display the result
payment_type_category_count

In [ ]:
### cell 25 ###

# we are shown the count under each category but it is better to have count% for comparison - create count_percent col
payment_type_category_count["count_percent"] = (
    payment_type_category_count["count"] / trip_data.shape[0]
) * 100
# print the data frame
payment_type_category_count

In [ ]:
### cell 26 ###

# let's see the number of categories available in both pickup and dropoff location - PULocationID and DOLocationID
(
    trip_data["PULocationID"].value_counts().shape,
    trip_data["DOLocationID"].value_counts().shape,
)

In [ ]:
### cell 27 ###

# Vectorized string concatenation using cuDF string methods
trip_data["routes"] = (
    trip_data["PULocationID"].astype("str")
        .str.cat(trip_data["DOLocationID"].astype("str"), sep="-")
)

In [ ]:
### cell 28 ###

trip_data.head()

In [ ]:
### cell 29 ###

# look into value_counts of 'passenger_count'
trip_data["passenger_count"].value_counts()

In [ ]:
### cell 30 ###

# restricted_fare_amount_data dataframe formation by filtering fare_amount less than 50 dollars
restricted_fare_amount_data = trip_data.loc[trip_data["fare_amount"] <= 50]
restricted_fare_amount_data.shape

In [ ]:
### cell 31 ###

# restricted_total_amount_data for filtering total_amount data to less than 50 dollars
restricted_total_amount_data = trip_data.loc[trip_data["total_amount"] <= 50]
restricted_total_amount_data.shape

In [ ]:
### cell 32 ###

restricted_tip_amount_data = trip_data.loc[trip_data["tip_amount"] < 10]
restricted_total_taxes_data = trip_data.loc[trip_data["total_taxes"] < 10]
restricted_tip_amount_data.shape, restricted_total_taxes_data.shape

In [ ]:
### cell 33 ###

# create a new series using value_counts() on 'PULocationID'
pickup_location_value_counts = trip_data["PULocationID"].value_counts()
# show the series
pickup_location_value_counts.head()

In [ ]:
### cell 34 ###

# top 10 frequent pickup locations using .nlargest(10).index
top_10_frequent_pickup_locations = pickup_location_value_counts.nlargest(10).index
top_10_frequent_pickup_locations

In [ ]:
### cell 35 ###

# create restricted_duration dataframe with .loc on 'duration' column
restricted_duration = trip_data.loc[trip_data["duration"] < 50]
restricted_duration.shape